# Capstone — mirrors your deployed research paper

This notebook contains the final data and claims that back the deployed ML-11 research paper.

## 1. Question

**Research Question**: Can we predict which pieces of content are entering a state of traffic decline based on their historical search visibility and staleness, before the traffic drops to zero?
**Decision Supported**: Prioritizing the content refresh and editorial update queue.

## 2. Data

**Dataset**: 30,000-row anonymized cross-sectional sample (`content_refresh_anonymized.csv`) representing a 90-day observation window.
**Warehouse Context**: Drawn from the larger FlyRank internship warehouse (v20260703, ~79 million daily performance rows across 17 months).
**Exclusions**: Pseudonymous identifiers (`client_id`, `content_id`) and label-derived metrics (`trend_direction`, `trend_pct`) were strictly excluded from features to prevent memorization and target leakage.

In [1]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"Rows: {len(df):,}")
print(f"Base Rate: {df['is_declining'].mean():.3f}")

## 3. Methodology

**Label**: `is_declining` = 1 when `trend_direction == "down"`.
**Baseline**: A hard-coded rule (stale + visible + striking distance).
**Validation**: Grouped client holdout split (`GroupShuffleSplit` by `client_id`) for primary evaluation. A secondary age-ordered (time-aware) split was used for temporal proxy validation.

## 4. Results (vs baseline)

**Same-Split Comparison (Grouped by Client)**:
- **Baseline**: Precision@50 = 0.400, ROC-AUC = 0.506
- **Random Forest**: Precision@50 = 0.560, ROC-AUC = 0.600
- **Logistic Regression**: Precision@50 = 0.820, ROC-AUC = 0.636

**Validation Strategy Audit (LogReg)**:
- **Grouped Split**: Precision@50 = 0.820
- **Age-Ordered / Time-Aware Split**: Precision@50 = 0.880

In [2]:
import pandas as pd
results = pd.DataFrame({
    'Model': ['Week 4 Baseline', 'Random Forest', 'Logistic Regression'],
    'Precision@50': [0.400, 0.560, 0.820],
    'ROC-AUC': [0.506, 0.600, 0.636]
})
results

## 5. Limitations

**Observational Data**: This data does not prove causality; updating a page is not mathematically guaranteed to recover traffic.
**No Semantic Features**: The model only sees numeric metadata (impressions, staleness), it cannot evaluate search intent shifts or factual inaccuracies. Human editorial judgment is still required.

## 6. Ranked recommendations

Based on ML-10 Action Playbook:
1. **High Priority Refresh**: High model probability of decline + strong historical visibility. Needs comprehensive review.
2. **Standard Refresh Review**: High decline risk but lower historical volume. Evaluate effort vs. reward.
3. **Preventative Update**: Stale (>180 days) but retain high traffic. Light refresh to prevent decay.

## 7. Artifacts the paper embeds

The following charts were generated in `scratch/chart_maker.py` and exported to `docs/assets/charts/`:
- `precision_comparison.png`
- `roc_auc_comparison.png`
- `validation_improvement.png`

## Self-check

- [x] Every section above is filled 
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`